In [1]:
%load_ext sql
%sql sqlite://

## step1: 
## read in csv of games to table

In [2]:
%%time
import csv
import sqlite3

%sql sqlite:///clasher.db
    
conn = sqlite3.connect("clasher.db")
cursor = conn.cursor()

with open('battlesStaging_12282020_WL_tagged.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    
    # Create table
    col_defs = ", ".join([f'"{col}" TEXT' for col in columns])
    cursor.execute(f"DROP TABLE IF EXISTS clasher")
    cursor.execute(f"CREATE TABLE clasher ({col_defs})")
    
    # Insert all rows
    placeholders = ", ".join(["?" for _ in columns])
    rows = [tuple(row[col] for col in columns) for row in reader]
    
    for row in rows:
        cursor.execute(f'INSERT INTO clasher VALUES ({placeholders})', row)

    conn.commit()

print("Done! Rows loaded:", cursor.execute("SELECT COUNT(*) FROM clasher").fetchone()[0])
print("First row:", cursor.execute("SELECT * FROM clasher").fetchone())

Done! Rows loaded: 1902766
First row: ('0', '2020-12-27 19:01:19+00:00', '54000050.0', '72000044.0', '5962.0', '#28JV2JQCV', '5952.0', '31.0', '1.0', '5832.0', '[3668, 765]', '#Y8GLLUQ', '16000014.0', '#2V28QCRY8', '5972.0', '-31.0', '0.0', '5832.0', '#8JGGJ8GU', '16000003.0', '[3668]', '', '26000009', '13', '26000042', '13', '28000007', '13', '26000013', '13', '26000048', '13', '26000025', '12', '26000037', '13', '28000001', '13', '[26000009, 26000013, 26000025, 26000037, 26000042, 26000048, 28000001, 28000007]', '103', '6', '0', '2', '2', '0', '3', '3', '4.375', '26000015', '13', '28000015', '13', '26000054', '13', '26000063', '13', '26000009', '13', '26000048', '13', '26000039', '13', '28000012', '13', '[26000009, 26000015, 26000039, 26000048, 26000054, 26000063, 28000012, 28000015]', '104', '6', '0', '2', '0', '1', '6', '1', '4.25')
CPU times: total: 1min 45s
Wall time: 2min 54s


In [3]:
cursor.execute("SELECT * FROM clasher")
print([desc[0] for desc in cursor.description])

['', 'battleTime', 'arena.id', 'gameMode.id', 'average.startingTrophies', 'winner.tag', 'winner.startingTrophies', 'winner.trophyChange', 'winner.crowns', 'winner.kingTowerHitPoints', 'winner.princessTowersHitPoints', 'winner.clan.tag', 'winner.clan.badgeId', 'loser.tag', 'loser.startingTrophies', 'loser.trophyChange', 'loser.crowns', 'loser.kingTowerHitPoints', 'loser.clan.tag', 'loser.clan.badgeId', 'loser.princessTowersHitPoints', 'tournamentTag', 'winner.card1.id', 'winner.card1.level', 'winner.card2.id', 'winner.card2.level', 'winner.card3.id', 'winner.card3.level', 'winner.card4.id', 'winner.card4.level', 'winner.card5.id', 'winner.card5.level', 'winner.card6.id', 'winner.card6.level', 'winner.card7.id', 'winner.card7.level', 'winner.card8.id', 'winner.card8.level', 'winner.cards.list', 'winner.totalcard.level', 'winner.troop.count', 'winner.structure.count', 'winner.spell.count', 'winner.common.count', 'winner.rare.count', 'winner.epic.count', 'winner.legendary.count', 'winner.e

## read in csv of cards

In [4]:
with open('CardMasterListSeason18_12082020.csv', 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    
    # Create table
    col_defs = ", ".join([f'"{col}" TEXT' for col in columns])
    cursor.execute("DROP TABLE IF EXISTS card_names")
    cursor.execute(f"CREATE TABLE card_names ({col_defs})")
    
    # Insert all rows
    placeholders = ", ".join(["?" for _ in columns])
    rows = [tuple(row[col] for col in columns) for row in reader]
    
    for row in rows:
        cursor.execute(f'INSERT INTO card_names VALUES ({placeholders})', row)
    conn.commit()

print("Done! Rows loaded:", cursor.execute("SELECT COUNT(*) FROM card_names").fetchone()[0])
print("First row:", cursor.execute("SELECT * FROM card_names").fetchone())
print("Columns:", cursor.execute("SELECT * FROM card_names").description)

Done! Rows loaded: 102
First row: ('26000000', 'Knight')
Columns: (('team.card1.id', None, None, None, None, None, None), ('team.card1.name', None, None, None, None, None, None))


## step 2: Join card names onto match records by replacing card IDs with their corresponding card names for all 8 winner and loser card slots.

In [5]:
%%time
cursor.execute("""
    CREATE TABLE clasher_new AS
    SELECT c.*,
        w1."team.card1.name" as "winner.card1.name",
        w2."team.card1.name" as "winner.card2.name",
        w3."team.card1.name" as "winner.card3.name",
        w4."team.card1.name" as "winner.card4.name",
        w5."team.card1.name" as "winner.card5.name",
        w6."team.card1.name" as "winner.card6.name",
        w7."team.card1.name" as "winner.card7.name",
        w8."team.card1.name" as "winner.card8.name",
        l1."team.card1.name" as "loser.card1.name",
        l2."team.card1.name" as "loser.card2.name",
        l3."team.card1.name" as "loser.card3.name",
        l4."team.card1.name" as "loser.card4.name",
        l5."team.card1.name" as "loser.card5.name",
        l6."team.card1.name" as "loser.card6.name",
        l7."team.card1.name" as "loser.card7.name",
        l8."team.card1.name" as "loser.card8.name"
    FROM clasher c
    LEFT JOIN card_names w1 ON c."winner.card1.id" = w1."team.card1.id"
    LEFT JOIN card_names w2 ON c."winner.card2.id" = w2."team.card1.id"
    LEFT JOIN card_names w3 ON c."winner.card3.id" = w3."team.card1.id"
    LEFT JOIN card_names w4 ON c."winner.card4.id" = w4."team.card1.id"
    LEFT JOIN card_names w5 ON c."winner.card5.id" = w5."team.card1.id"
    LEFT JOIN card_names w6 ON c."winner.card6.id" = w6."team.card1.id"
    LEFT JOIN card_names w7 ON c."winner.card7.id" = w7."team.card1.id"
    LEFT JOIN card_names w8 ON c."winner.card8.id" = w8."team.card1.id"
    LEFT JOIN card_names l1 ON c."loser.card1.id" = l1."team.card1.id"
    LEFT JOIN card_names l2 ON c."loser.card2.id" = l2."team.card1.id"
    LEFT JOIN card_names l3 ON c."loser.card3.id" = l3."team.card1.id"
    LEFT JOIN card_names l4 ON c."loser.card4.id" = l4."team.card1.id"
    LEFT JOIN card_names l5 ON c."loser.card5.id" = l5."team.card1.id"
    LEFT JOIN card_names l6 ON c."loser.card6.id" = l6."team.card1.id"
    LEFT JOIN card_names l7 ON c."loser.card7.id" = l7."team.card1.id"
    LEFT JOIN card_names l8 ON c."loser.card8.id" = l8."team.card1.id"
""")

# 4. Replace clasher with the new table
cursor.execute("DROP TABLE clasher")
cursor.execute("ALTER TABLE clasher_new RENAME TO clasher")
conn.commit()

print("Done!", cursor.execute("SELECT COUNT(*) FROM clasher").fetchone()[0], "rows")
print("First row:", cursor.execute("SELECT * FROM clasher").fetchone())

Done! 1902766 rows
First row: ('0', '2020-12-27 19:01:19+00:00', '54000050.0', '72000044.0', '5962.0', '#28JV2JQCV', '5952.0', '31.0', '1.0', '5832.0', '[3668, 765]', '#Y8GLLUQ', '16000014.0', '#2V28QCRY8', '5972.0', '-31.0', '0.0', '5832.0', '#8JGGJ8GU', '16000003.0', '[3668]', '', '26000009', '13', '26000042', '13', '28000007', '13', '26000013', '13', '26000048', '13', '26000025', '12', '26000037', '13', '28000001', '13', '[26000009, 26000013, 26000025, 26000037, 26000042, 26000048, 28000001, 28000007]', '103', '6', '0', '2', '2', '0', '3', '3', '4.375', '26000015', '13', '28000015', '13', '26000054', '13', '26000063', '13', '26000009', '13', '26000048', '13', '26000039', '13', '28000012', '13', '[26000009, 26000015, 26000039, 26000048, 26000054, 26000063, 28000012, 28000015]', '104', '6', '0', '2', '0', '1', '6', '1', '4.25', 'Golem', 'Electro Wizard', 'Lightning', 'Bomber', 'Night Witch', 'Guards', 'Inferno Dragon', 'Arrows', 'Baby Dragon', 'Barbarian Barrel', 'Cannon Cart', 'Elect

## step 3: unpivot all 16 card slots into rows and aggregate win/loss counts per card, grouped by card name

In [6]:
%%time
cursor.execute("DROP TABLE IF EXISTS card_stats")
cursor.execute("""
    CREATE TABLE card_stats AS
    SELECT card_name, SUM(wins) as win_count, SUM(losses) as loss_count
    FROM (
        SELECT "winner.card1.name" as card_name, 1 as wins, 0 as losses FROM clasher UNION ALL
        SELECT "winner.card2.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card3.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card4.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card5.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card6.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card7.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card8.name", 1, 0 FROM clasher UNION ALL
        SELECT "loser.card1.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card2.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card3.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card4.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card5.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card6.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card7.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card8.name", 0, 1 FROM clasher
    )
    WHERE card_name IS NOT NULL
    GROUP BY card_name
    ORDER BY win_count DESC
""")
conn.commit()

cursor.execute("SELECT * FROM card_stats LIMIT 10").fetchall()

CPU times: total: 1min 44s
Wall time: 1min 49s


[('Wizard', 582969, 583754),
 ('Valkyrie', 570154, 558447),
 ('The Log', 542295, 531042),
 ('Zap', 524471, 501257),
 ('Skeleton Army', 517161, 510175),
 ('Fireball', 477821, 485420),
 ('Hog Rider', 413462, 399929),
 ('Arrows', 397171, 394434),
 ('Mega Knight', 388330, 387939),
 ('Baby Dragon', 360764, 355725)]

## step 4: add new column "win_ratio"

In [7]:
%%time
cursor.execute("ALTER TABLE card_stats ADD COLUMN win_ratio REAL")
cursor.execute("UPDATE card_stats SET win_ratio = ROUND(win_count * 1.0 / (win_count + loss_count), 3)")
conn.commit()

CPU times: total: 0 ns
Wall time: 0 ns


In [8]:
results = cursor.execute("""
    SELECT * FROM card_stats 
    ORDER BY win_ratio DESC 
""").fetchall()

for row in results:
    print(row)

('Mega Minion', 111710, 103675, 0.519)
('Battle Ram', 57923, 53903, 0.518)
('Lava Hound', 37663, 35369, 0.516)
('Dark Prince', 132038, 125118, 0.513)
('Goblin Gang', 258005, 246111, 0.512)
('Skeleton Barrel', 119275, 113771, 0.512)
('Night Witch', 107716, 102496, 0.512)
('Rascals', 20830, 19848, 0.512)
('Zap', 524471, 501257, 0.511)
('Barbarian Barrel', 146343, 140114, 0.511)
('Golem', 133822, 127841, 0.511)
('Goblin Barrel', 317905, 305331, 0.51)
('Mini P.E.K.K.A', 302441, 291910, 0.509)
('P.E.K.K.A', 215439, 208025, 0.509)
('Lightning', 116650, 112431, 0.509)
('Hunter', 104937, 101168, 0.509)
('Flying Machine', 40887, 39519, 0.509)
('Bowler', 34572, 33388, 0.509)
('Cannon Cart', 32392, 31192, 0.509)
('Hog Rider', 413462, 399929, 0.508)
('Goblin Giant', 14406, 13968, 0.508)
('Bandit', 168286, 163589, 0.507)
('Poison', 137213, 134150, 0.506)
('Royal Giant', 120295, 117589, 0.506)
('Valkyrie', 570154, 558447, 0.505)
('The Log', 542295, 531042, 0.505)
('Balloon', 282273, 276990, 0.505)
(